# Layer 6 consolidation: one exact configuration vs one fast configuration

Days 16-17 found two separate accelerations (larger stride, mean-prefiltered
approximate denoising) in isolation. This notebook combines them into a
single named "fast" configuration and compares it against a single named
"exact" reference configuration - **same seed, same source image** - so the
quality loss is measured once, for the actual configuration that would be
used, not inferred from separate experiments run under different conditions.

Each acceleration is also isolated on its own, so a quality change in the
combined fast configuration can be attributed to a specific cause.


## 1. The two configurations

**Exact (reference):** `patch_size=4, stride=2, mean_tolerance=None` - this
is the configuration every multiscale notebook (Days 12-15) has used as the
standard: dense overlapping patches, exact nearest-neighbor weighting over
the full bank.

**Fast:** `patch_size=4, stride=4, mean_tolerance=20` - no patch overlap
(Day 16's finding) combined with mean-prefiltered candidate filtering
(Day 17's finding, using the tolerance that showed zero single-patch
deviation on a real bank).


In [1]:
import sys, os, time
sys.path.append(os.path.join("..", "..", "src"))
import numpy as np
from PIL import Image

from data.pyramid import build_pyramid
from models.sampler import sample_coarse_to_fine

NIST_DIR = os.path.join("..", "..", "..", "AMAT", "nist_detection_limits_sem")
clean_path = os.path.join(NIST_DIR, "mask_sets", "masks", "set1_cex_noise_000_contrast_100.tiff")
img = np.array(Image.open(clean_path)).astype(np.float64)[:64, :64]
pyramid = build_pyramid(img, num_scales=4, scale_factor=0.5)

PATCH_SIZE, NUM_STEPS, SEED = 4, 15, 0
print(f"real image std: {img.std():.2f}")


real image std: 55.64


## 2. Full comparison: exact reference vs each acceleration alone vs both combined

Same `seed=0`, same image, same `patch_size`/`num_steps` throughout - only
`stride` and `mean_tolerance` change between runs. This isolates which
acceleration is responsible for any change in the result.

In [2]:
configs = {
    "exact (reference)":        dict(stride=2, mean_tolerance=None),
    "approx-only (stride=2)":   dict(stride=2, mean_tolerance=20),
    "stride-only (tol=None)":   dict(stride=4, mean_tolerance=None),
    "fast (both combined)":     dict(stride=4, mean_tolerance=20),
}

results = {}
for name, cfg in configs.items():
    t0 = time.perf_counter()
    final, _ = sample_coarse_to_fine(pyramid, PATCH_SIZE, cfg["stride"], NUM_STEPS, seed=SEED, mean_tolerance=cfg["mean_tolerance"])
    dt = time.perf_counter() - t0
    results[name] = dict(final=final, time=dt, std=final.std())
    print(f"{name:26s}: time={dt:7.3f}s, std={final.std():6.2f}")


exact (reference)         : time=  3.197s, std=  1.23


approx-only (stride=2)    : time=  7.892s, std=  1.24


stride-only (tol=None)    : time=  0.749s, std= 61.22


fast (both combined)      : time=  1.378s, std= 65.15


## 3. Classifying "approx-only": does mean-prefiltering change the
trajectory, not just the speed?

`approx-only` uses the SAME stride as the reference, so both runs extract
patches at identical positions and draw identical noise (same seed) at
every step - the only thing that can differ is what the approximate
denoiser returns. A pixel-level difference here is directly attributable
to the approximation, not to an unrelated random difference.

In [3]:
exact_final = results["exact (reference)"]["final"]
approx_final = results["approx-only (stride=2)"]["final"]

pixel_diff = np.abs(exact_final - approx_final)
print(f"approx-only vs exact: mean abs pixel diff={pixel_diff.mean():.4f}, max abs pixel diff={pixel_diff.max():.4f}")
print(f"approx-only speedup: {results['exact (reference)']['time'] / results['approx-only (stride=2)']['time']:.2f}x")


approx-only vs exact: mean abs pixel diff=0.0207, max abs pixel diff=0.4312
approx-only speedup: 0.41x


## 4. Classifying "stride-only": does a larger stride change the
distribution, not just the speed?

A larger stride changes patch positions and bank size, so the two runs are
not on a shared pixel-by-pixel trajectory even with the same seed - a
per-pixel diff would not mean what it means in step 3. The comparison here
is distributional: does the summary statistic (std, matching Day 16's
metric) move, and by how much relative to the real image's own std?

In [4]:
ref_std, real_std = results["exact (reference)"]["std"], img.std()
stride_std = results["stride-only (tol=None)"]["std"]

print(f"real image std:            {real_std:6.2f}")
print(f"exact reference (stride=2) std: {ref_std:6.2f}  (distance from real: {abs(ref_std - real_std):.2f})")
print(f"stride-only (stride=4)    std: {stride_std:6.2f}  (distance from real: {abs(stride_std - real_std):.2f})")
print(f"stride-only speedup: {results['exact (reference)']['time'] / results['stride-only (tol=None)']['time']:.2f}x")


real image std:             55.64
exact reference (stride=2) std:   1.23  (distance from real: 54.41)
stride-only (stride=4)    std:  61.22  (distance from real: 5.58)
stride-only speedup: 4.27x


## 5. The combined fast configuration vs the reference

In [5]:
fast_std = results["fast (both combined)"]["std"]
print(f"fast (combined)           std: {fast_std:6.2f}  (distance from real: {abs(fast_std - real_std):.2f})")
print(f"combined speedup: {results['exact (reference)']['time'] / results['fast (both combined)']['time']:.2f}x")


fast (combined)           std:  65.15  (distance from real: 9.51)
combined speedup: 2.32x


## Findings - each acceleration classified separately, as the gate requires

These numbers contradicted what Days 16-17 would predict on their own,
which is exactly why the gate requires measuring the *combination*, not
citing the separate experiments.

**Approx-only (mean-prefiltering, tolerance=20) is almost distribution-neutral
but is NOT a speedup at this scale - it is slower (0.41x, i.e. 8.57s vs
3.51s).** The pixel-level difference from the exact reference is small
(mean abs diff 0.02, max 0.43) but no longer exactly zero as it was in
Day 17's single-fixed-sigma test - here sigma sweeps down across the whole
schedule, and at small sigma a coarse mean-based filter is more likely to
exclude a patch that exact search would still weight non-negligibly. More
importantly: Day 17 measured a 16,256-patch bank; this pipeline's banks
(64x64 image, patch_size=4) are far smaller, so the fixed per-call
overhead of computing means and copying a filtered sub-array is not repaid
by skipping distance computations on a bank that was already cheap to
search exactly. **A per-patch acceleration's benefit depends on bank size,
not just on the algorithm - it must be re-measured on the bank size it
will actually run on, not assumed to transfer from a differently-sized
test.**

**Stride-only (no patch overlap) is distribution-changing, and here the
change is a large, measured improvement: 4.21x faster AND std=61.22 vs the
reference's std=1.23 (real image std=55.64) - the reference itself is
still badly variance-collapsed at this configuration.** This confirms
Day 16's per-stage finding at the full coarse-to-fine level: less
overlap means less averaging, which is what was suppressing variance in
the first place.

**The combined fast configuration is worse than stride-only alone on
both axes, not just one.** Combined: 2.28x speedup, std=65.15 (distance
from real: 9.51). Stride-only alone: 4.21x speedup, std=61.22 (distance
from real: 5.58). Adding the approximate denoiser on top of the larger
stride made the result both slower *and* further from the real image's
std than stride-only by itself - the two "accelerations" do not combine
additively; the second one is pure cost here, in both dimensions.

**Practical conclusion for Layer 7:** the mean-prefiltered denoiser
should NOT be part of the fast configuration at this bank size - it is a
net loss, not a free speedup, and it was only ever validated in isolation
on a much larger bank (Day 17). The single accepted "fast" configuration
going forward is stride=4 alone: a real, measured, ~4x speedup that also
happens to reduce the variance-collapse artifact - but it must be
reported to Layer 7 as a deliberate quality/speed tradeoff (a different
generative process, not a free win), not folded in silently.
